# MALTO — v3 Fast (uses uploaded model)

**Goal:** Beat 0.96423 (current 1st place)

**Changes from v1 (0.95341):**
1. SWA: top-3 checkpoint averaging per fold
2. Per-class ensemble weights (removes SVC damage on DeepSeek)
3. Wider threshold search [0.60, 1.50]
4. Skip full-data training — use uploaded model
5. 3-fold (fits in 1:30h budget)

**Runtime:** ~70 min on T4×2

In [ ]:
import os, warnings, gc, time, json, re, pickle
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler

from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from scipy.sparse import hstack as sparse_hstack
from scipy.optimize import minimize
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

SEED = 42
NUM_LABELS = 6
LABEL_MAP = {0: 'Human', 1: 'DeepSeek', 2: 'Grok', 3: 'Claude', 4: 'Gemini', 5: 'ChatGPT'}

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    torch.cuda.manual_seed_all(SEED)
    N_GPUS  = torch.cuda.device_count()
    USE_AMP = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32       = False
    print(f'GPU: {torch.cuda.get_device_name(0)} x {N_GPUS}')
    print(f'AMP: {USE_AMP} | TF32: disabled')
else:
    DEVICE, N_GPUS, USE_AMP = torch.device('cpu'), 1, False

KAGGLE_PATHS = [
    '/kaggle/input/malto-recruitment-hackathon',
    '/kaggle/input/test-and-train',
    '/kaggle/input/datasets/aliivaezii/test-and-train',
    '/kaggle/input', '.', '..'
]
DATA_DIR = None
for p in KAGGLE_PATHS:
    if os.path.exists(os.path.join(p, 'train.csv')):
        DATA_DIR = p
        break
if DATA_DIR is None and os.path.isdir('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.csv' in files:
            DATA_DIR = root
            break
assert DATA_DIR is not None, f'train.csv not found!'

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
train_df['TEXT'] = train_df['TEXT'].fillna('empty')
test_df['TEXT']  = test_df['TEXT'].fillna('empty')

y_all       = train_df['LABEL'].values
texts_train = train_df['TEXT'].values
texts_test  = test_df['TEXT'].values

print(f'Data dir: {DATA_DIR}')
print(f'Device: {DEVICE} | GPUs: {N_GPUS}')
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

In [ ]:
# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = 'answerdotai/ModernBERT-base'
MAX_LEN     = 512
BATCH_SIZE  = 16 * N_GPUS
EPOCHS      = 10
LR          = 3e-5
PATIENCE    = 3
N_FOLDS     = 3       # 3-fold to fit in 1:30h budget
NUM_WORKERS = 4
TOP_K_CKPTS = 3       # SWA: average top-3 checkpoints
DRW_CAP     = 10      # REVERTED from 20 (v2 showed 20 is too aggressive)

# Uploaded model (skip full-data training)
UPLOADED_MODEL_DIR = '/kaggle/input/models/aliivaezii/malto-1st-place-model/other/default/1'
ALPHA = 0.5  # fold_avg vs uploaded model blend (0.5 = equal weight)

print(f'Model: {MODEL_NAME}')
print(f'Batch: {BATCH_SIZE} | Folds: {N_FOLDS} | Epochs: {EPOCHS} | Patience: {PATIENCE}')
print(f'SWA: top-{TOP_K_CKPTS} | DRW cap: {DRW_CAP}x')
print(f'Uploaded model: {UPLOADED_MODEL_DIR}')
print(f'Uploaded model exists: {os.path.isdir(UPLOADED_MODEL_DIR)}')
if os.path.isdir(UPLOADED_MODEL_DIR):
    print(f'  Files: {os.listdir(UPLOADED_MODEL_DIR)}')

In [ ]:
# ============================================================
# LOSS: LDAM + Gradual DRW + Label Smoothing
# ============================================================
class LDAMLoss(nn.Module):
    def __init__(self, class_counts, max_margin=0.5,
                 drw_epoch=1, drw_ramp_epochs=3, label_smoothing=0.1):
        super().__init__()
        safe = np.maximum(class_counts, 1).astype(np.float64)
        margins = np.clip(max_margin / np.power(safe, 0.25), 0, 1.0)
        self.register_buffer('margins', torch.tensor(margins, dtype=torch.float32))
        self.drw_epoch = drw_epoch
        self.drw_ramp_epochs = drw_ramp_epochs
        self.current_epoch = 0
        self.label_smoothing = label_smoothing
        inv = 1.0 / safe
        w = inv / inv.sum() * len(class_counts)
        w = np.clip(w, w.min(), w.min() * DRW_CAP)
        w = w / w.sum() * len(class_counts)
        w = np.nan_to_num(w, nan=1.0, posinf=1.0, neginf=1.0)
        self.register_buffer('drw_weights', torch.tensor(w, dtype=torch.float32))

    def set_epoch(self, epoch):
        self.current_epoch = epoch

    def _blended_weight(self, device):
        ep = self.current_epoch
        if ep < self.drw_epoch:
            return None
        t = min((ep - self.drw_epoch) / max(self.drw_ramp_epochs, 1), 1.0)
        ones = torch.ones_like(self.drw_weights)
        return (ones + t * (self.drw_weights - ones)).to(device)

    def forward(self, logits, targets):
        logits = logits.float()
        if torch.isnan(logits).any():
            logits = torch.nan_to_num(logits, nan=0.0)
        margins = self.margins.to(logits.device)[targets]
        adjusted = logits.clone()
        adjusted[torch.arange(len(targets), device=logits.device), targets] -= margins
        weight = self._blended_weight(logits.device)
        loss = F.cross_entropy(adjusted, targets, weight=weight,
                               label_smoothing=self.label_smoothing)
        if torch.isnan(loss):
            return F.cross_entropy(logits, targets)
        return loss

print(f'LDAMLoss ready (DRW {DRW_CAP}x cap)')

In [ ]:
# ============================================================
# TOKENIZATION
# ============================================================
print('Tokenizing...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_enc = tokenizer(list(texts_train), padding='max_length', truncation=True,
                      max_length=MAX_LEN, return_tensors='pt')
test_enc = tokenizer(list(texts_test), padding='max_length', truncation=True,
                     max_length=MAX_LEN, return_tensors='pt')
print(f'Train: {train_enc["input_ids"].shape}, Test: {test_enc["input_ids"].shape}')

In [ ]:
# ============================================================
# DATASET & OPTIMIZER
# ============================================================
class TextDataset(Dataset):
    def __init__(self, encodings, labels=None, indices=None):
        self.encodings = encodings
        self.labels = labels
        self.indices = indices
    def __len__(self):
        return len(self.indices) if self.indices is not None else len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        real_idx = self.indices[idx] if self.indices is not None else idx
        item = {k: v[real_idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[real_idx], dtype=torch.long)
        return item

def get_optimizer(model, lr, llrd=0.9):
    import re as _re
    no_decay = ['bias', 'LayerNorm', 'layernorm', 'layer_norm']
    params = []
    head = [(n, p) for n, p in model.named_parameters() if 'classifier' in n or 'head' in n]
    if head:
        params.append({'params': [p for n, p in head if not any(nd in n for nd in no_decay)], 'lr': lr, 'weight_decay': 0.01})
        params.append({'params': [p for n, p in head if any(nd in n for nd in no_decay)], 'lr': lr, 'weight_decay': 0.0})
    all_names = [n for n, _ in model.named_parameters()]
    layer_nums = set()
    for n in all_names:
        m = _re.search(r'(?:encoder\.layer|layers)\.(\d+)\.', n)
        if m: layer_nums.add(int(m.group(1)))
    num_layers = max(layer_nums) + 1 if layer_nums else 12
    for i in range(num_layers - 1, -1, -1):
        layer_lr = lr * (llrd ** (num_layers - 1 - i))
        patterns = [f'encoder.layer.{i}.', f'layers.{i}.']
        lp = [(n, p) for n, p in model.named_parameters() if any(pat in n for pat in patterns)]
        if lp:
            wd_p = [p for n, p in lp if not any(nd in n for nd in no_decay)]
            nowd_p = [p for n, p in lp if any(nd in n for nd in no_decay)]
            if wd_p: params.append({'params': wd_p, 'lr': layer_lr, 'weight_decay': 0.01})
            if nowd_p: params.append({'params': nowd_p, 'lr': layer_lr, 'weight_decay': 0.0})
    emb_lr = lr * (llrd ** num_layers)
    emb_params = [(n, p) for n, p in model.named_parameters() if 'embed' in n.lower()]
    if emb_params:
        wd_p = [p for n, p in emb_params if not any(nd in n for nd in no_decay)]
        nowd_p = [p for n, p in emb_params if any(nd in n for nd in no_decay)]
        if wd_p: params.append({'params': wd_p, 'lr': emb_lr, 'weight_decay': 0.01})
        if nowd_p: params.append({'params': nowd_p, 'lr': emb_lr, 'weight_decay': 0.0})
    assigned = set(id(p) for grp in params for p in grp['params'])
    rem = [p for n, p in model.named_parameters() if id(p) not in assigned and p.requires_grad]
    if rem: params.append({'params': rem, 'lr': lr * 0.5, 'weight_decay': 0.01})
    return torch.optim.AdamW(params)

print('Dataset & optimizer ready.')

In [ ]:
# ============================================================
# K-FOLD TRAINING (SWA top-K checkpoint averaging)
# ============================================================
def train_fold(fold_idx, train_idx, val_idx):
    gc.collect(); torch.cuda.empty_cache()
    print(f'\n{"="*50}')
    print(f'FOLD {fold_idx+1}/{N_FOLDS} | Train: {len(train_idx)}, Val: {len(val_idx)}')
    print(f'{"="*50}')

    train_loader = DataLoader(TextDataset(train_enc, y_all, train_idx),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(TextDataset(train_enc, y_all, val_idx),
        batch_size=BATCH_SIZE*2, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(TextDataset(test_enc),
        batch_size=BATCH_SIZE*2, num_workers=NUM_WORKERS, pin_memory=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True)
    if N_GPUS > 1: model = nn.DataParallel(model)
    model.to(DEVICE)

    fold_counts = np.bincount(y_all[train_idx], minlength=NUM_LABELS).astype(float)
    criterion = LDAMLoss(fold_counts, max_margin=0.5, drw_epoch=1, drw_ramp_epochs=3)
    criterion.to(DEVICE)
    print(f'  DRW weights: {criterion.drw_weights.cpu().numpy().round(2)}')

    actual_model = model.module if hasattr(model, 'module') else model
    optimizer = get_optimizer(actual_model, lr=LR, llrd=0.9)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_scheduler('cosine', optimizer,
        num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)
    scaler = GradScaler(enabled=USE_AMP)

    top_ckpts = []
    best_f1 = 0
    patience_counter = 0

    for epoch in range(EPOCHS):
        t0 = time.time()
        criterion.set_epoch(epoch)
        if epoch < criterion.drw_epoch: drw_pct = '  0%'
        else:
            t = min((epoch - criterion.drw_epoch) / criterion.drw_ramp_epochs, 1.0)
            drw_pct = f'{int(t*100):3d}%'

        model.train()
        total_loss, valid_steps = 0.0, 0
        for batch in tqdm(train_loader, desc=f'E{epoch+1}', leave=False):
            optimizer.zero_grad()
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = inputs.pop('labels')
            with autocast(enabled=USE_AMP):
                outputs = model(**inputs)
                loss = criterion(outputs.logits, labels)
            if not (torch.isnan(loss) or torch.isinf(loss)):
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()
                valid_steps += 1
            scheduler.step()
        avg_loss = total_loss / max(valid_steps, 1)

        model.eval()
        preds, labels_list = [], []
        with torch.no_grad():
            for batch in val_loader:
                inputs = {k: v.to(DEVICE) for k, v in batch.items()}
                labs = inputs.pop('labels')
                outputs = model(**inputs)
                preds.extend(outputs.logits.argmax(-1).cpu().numpy())
                labels_list.extend(labs.cpu().numpy())
        val_f1 = f1_score(labels_list, preds, average='macro')
        elapsed = time.time() - t0

        cur_model = model.module if hasattr(model, 'module') else model
        state = {k: v.cpu().clone() for k, v in cur_model.state_dict().items()}
        if len(top_ckpts) < TOP_K_CKPTS or val_f1 > top_ckpts[-1][0]:
            if len(top_ckpts) >= TOP_K_CKPTS: top_ckpts.pop()
            top_ckpts.append((val_f1, state))
            top_ckpts.sort(key=lambda x: x[0], reverse=True)

        if val_f1 > best_f1:
            best_f1, patience_counter = val_f1, 0
            marker = ' ** BEST **'
        else:
            patience_counter += 1
            marker = f' (p={patience_counter}/{PATIENCE})'
        print(f'  E{epoch+1} | loss={avg_loss:.4f} | val_f1={val_f1:.4f} | DRW={drw_pct} | {elapsed:.0f}s{marker}')
        if patience_counter >= PATIENCE:
            print(f'  Early stop at E{epoch+1}')
            break

    n_avg = len(top_ckpts)
    print(f'  Averaging top-{n_avg}: {[f"{f:.4f}" for f, _ in top_ckpts]}')
    avg_state = {}
    for key in top_ckpts[0][1]:
        avg_state[key] = (sum(c[key].float() for _, c in top_ckpts) / n_avg).to(top_ckpts[0][1][key].dtype)
    cur_model = model.module if hasattr(model, 'module') else model
    cur_model.load_state_dict(avg_state)
    model.eval()

    val_logits, test_logits = [], []
    with torch.no_grad():
        for batch in val_loader:
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            inputs.pop('labels', None)
            val_logits.extend(model(**inputs).logits.float().cpu().numpy())
        for batch in test_loader:
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            test_logits.extend(model(**inputs).logits.float().cpu().numpy())

    del model, optimizer, scheduler, scaler, top_ckpts, avg_state
    gc.collect(); torch.cuda.empty_cache()
    return np.array(val_logits), np.array(test_logits), best_f1

print(f'Training ready | DRW {DRW_CAP}x | SWA top-{TOP_K_CKPTS} | {N_FOLDS}-fold')

In [ ]:
# ============================================================
# RUN K-FOLD
# ============================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_logits = np.zeros((len(y_all), NUM_LABELS))
test_logits_sum = np.zeros((len(test_df), NUM_LABELS))
fold_scores = []
t_start = time.time()

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
    val_logits, test_logits, best_f1 = train_fold(fold_idx, train_idx, val_idx)
    oof_logits[val_idx] = val_logits
    test_logits_sum += test_logits
    fold_scores.append(best_f1)
    print(f'  Fold {fold_idx+1} best: {best_f1:.4f} | Time: {(time.time()-t_start)/60:.1f}min')

test_logits_avg = test_logits_sum / N_FOLDS
print(f'\n{"="*50}')
print(f'{N_FOLDS}-FOLD COMPLETE')
print(f'  Scores: {[f"{s:.4f}" for s in fold_scores]}')
print(f'  Mean: {np.mean(fold_scores):.4f} \u00b1 {np.std(fold_scores):.4f}')
print(f'  Time: {(time.time()-t_start)/60:.1f} min')
print(f'{"="*50}')

In [ ]:
# ============================================================
# TEMPERATURE SCALING
# ============================================================
logits_t = torch.tensor(oof_logits, dtype=torch.float32)
labels_t = torch.tensor(y_all, dtype=torch.long)
best_temp, best_nll = 1.0, float('inf')
for temp in np.arange(0.3, 3.0, 0.05):
    nll = F.cross_entropy(logits_t / temp, labels_t).item()
    if nll < best_nll: best_nll, best_temp = nll, temp

oof_probs = torch.softmax(logits_t / best_temp, dim=-1).numpy()
test_probs = torch.softmax(torch.tensor(test_logits_avg) / best_temp, dim=-1).numpy()

oof_f1 = f1_score(y_all, oof_probs.argmax(1), average='macro')
print(f'Temperature: {best_temp:.2f} | OOF F1: {oof_f1:.4f}')
print(classification_report(y_all, oof_probs.argmax(1),
    target_names=list(LABEL_MAP.values())))

In [ ]:
# ============================================================
# TF-IDF + CALIBRATED SVC
# ============================================================
print('Building TF-IDF + SVC...')
t_svc = time.time()

char_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5),
    max_features=50000, min_df=2, max_df=0.95, sublinear_tf=True)
word_tfidf = TfidfVectorizer(analyzer='word', ngram_range=(1,2),
    max_features=50000, min_df=2, max_df=0.95, sublinear_tf=True)

char_train = char_tfidf.fit_transform(texts_train)
char_test  = char_tfidf.transform(texts_test)
word_train = word_tfidf.fit_transform(texts_train)
word_test  = word_tfidf.transform(texts_test)
X_sparse_train = sparse_hstack([char_train, word_train])
X_sparse_test  = sparse_hstack([char_test,  word_test])

svc_oof_probs = np.zeros((len(y_all), NUM_LABELS))
svc_test_probs_sum = np.zeros((len(texts_test), NUM_LABELS))

# Use SAME fold splits as ModernBERT for aligned OOF
skf_svc = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold_idx, (tr_idx, va_idx) in enumerate(skf_svc.split(np.zeros(len(y_all)), y_all)):
    svc = CalibratedClassifierCV(
        LinearSVC(C=5.0, class_weight='balanced', max_iter=20000), cv=3, method='sigmoid')
    svc.fit(X_sparse_train[tr_idx], y_all[tr_idx])
    svc_oof_probs[va_idx] = svc.predict_proba(X_sparse_train[va_idx])
    svc_test_probs_sum += svc.predict_proba(X_sparse_test)
    fold_f1 = f1_score(y_all[va_idx], svc_oof_probs[va_idx].argmax(1), average='macro')
    print(f'  SVC Fold {fold_idx+1}: F1={fold_f1:.4f}')

svc_test_probs = svc_test_probs_sum / N_FOLDS
svc_oof_f1 = f1_score(y_all, svc_oof_probs.argmax(1), average='macro')
print(f'\nSVC OOF F1: {svc_oof_f1:.4f}  ({time.time()-t_svc:.0f}s)')
print(classification_report(y_all, svc_oof_probs.argmax(1),
    target_names=list(LABEL_MAP.values())))

In [ ]:
# ============================================================
# LOAD UPLOADED MODEL + INFERENCE (replaces full-data training)
# ============================================================
print(f'Loading uploaded model from: {UPLOADED_MODEL_DIR}')

uploaded_model = AutoModelForSequenceClassification.from_pretrained(
    UPLOADED_MODEL_DIR, num_labels=NUM_LABELS, ignore_mismatched_sizes=True)
if N_GPUS > 1:
    uploaded_model = nn.DataParallel(uploaded_model)
uploaded_model.to(DEVICE)
uploaded_model.eval()

test_loader = DataLoader(TextDataset(test_enc),
    batch_size=BATCH_SIZE*2, num_workers=NUM_WORKERS, pin_memory=True)

uploaded_test_logits = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Uploaded model inference'):
        inputs = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = uploaded_model(**inputs)
        uploaded_test_logits.extend(outputs.logits.float().cpu().numpy())

uploaded_test_logits = np.array(uploaded_test_logits)
full_test_probs = torch.softmax(
    torch.tensor(uploaded_test_logits, dtype=torch.float32) / best_temp, dim=-1).numpy()

del uploaded_model
gc.collect(); torch.cuda.empty_cache()

print(f'Uploaded model inference done.')
print(f'Test preds: {dict(zip(LABEL_MAP.values(), [(full_test_probs.argmax(1)==i).sum() for i in range(NUM_LABELS)]))}')
print(f'Total elapsed: {(time.time()-t_start)/60:.1f} min')

In [ ]:
# ============================================================
# ENSEMBLE: Global + Per-class Nelder-Mead
# ============================================================
print('Ensemble weight optimization...')

# ── Strategy 1: Global weights ──
def neg_macro_f1(weights, probs_list, labels):
    w = np.abs(weights); w = w / w.sum()
    blended = sum(wi * pi for wi, pi in zip(w, probs_list))
    return -f1_score(labels, blended.argmax(1), average='macro')

oof_components = [oof_probs, svc_oof_probs]
best_result, best_neg_f1 = None, 0
for init in [[0.85,0.15],[0.80,0.20],[0.90,0.10],[0.75,0.25],[0.70,0.30],[0.95,0.05]]:
    result = minimize(neg_macro_f1, init, args=(oof_components, y_all),
        method='Nelder-Mead', options={'maxiter': 5000, 'xatol': 1e-5, 'fatol': 1e-6})
    if best_result is None or result.fun < best_neg_f1:
        best_neg_f1, best_result = result.fun, result

opt_w = np.abs(best_result.x) / np.abs(best_result.x).sum()
oof_blend_global = opt_w[0] * oof_probs + opt_w[1] * svc_oof_probs
global_f1 = f1_score(y_all, oof_blend_global.argmax(1), average='macro')
print(f'  Global: MB={opt_w[0]:.3f} SVC={opt_w[1]:.3f} -> OOF F1={global_f1:.4f}')

# ── Strategy 2: Per-class SVC weights ──
def neg_f1_pc(w_svc, mb_probs, svc_probs, labels):
    w = np.clip(w_svc, 0.0, 0.5)
    blended = np.zeros_like(mb_probs)
    for c in range(NUM_LABELS):
        blended[:, c] = (1 - w[c]) * mb_probs[:, c] + w[c] * svc_probs[:, c]
    return -f1_score(labels, blended.argmax(1), average='macro')

np.random.seed(SEED)
best_pc = None
for _ in range(12):
    result = minimize(neg_f1_pc, np.random.uniform(0.05, 0.45, NUM_LABELS),
        args=(oof_probs, svc_oof_probs, y_all),
        method='Nelder-Mead', options={'maxiter': 10000, 'xatol': 1e-5, 'fatol': 1e-7})
    if best_pc is None or result.fun < best_pc.fun: best_pc = result

opt_w_pc = np.clip(best_pc.x, 0.0, 0.5)
oof_blend_pc = np.zeros_like(oof_probs)
for c in range(NUM_LABELS):
    oof_blend_pc[:, c] = (1-opt_w_pc[c]) * oof_probs[:, c] + opt_w_pc[c] * svc_oof_probs[:, c]
pc_f1 = f1_score(y_all, oof_blend_pc.argmax(1), average='macro')
print(f'  Per-class: {dict(zip(LABEL_MAP.values(), opt_w_pc.round(3)))} -> OOF F1={pc_f1:.4f}')

# Pick better strategy
use_perclass = pc_f1 >= global_f1
if use_perclass:
    oof_blend, oof_blend_f1 = oof_blend_pc, pc_f1
    print(f'  >> PER-CLASS (+{pc_f1-global_f1:.4f})')
else:
    oof_blend, oof_blend_f1 = oof_blend_global, global_f1
    print(f'  >> GLOBAL')

# Build test blend
mbert_test = ALPHA * test_probs + (1-ALPHA) * full_test_probs
if use_perclass:
    test_blend = np.zeros_like(mbert_test)
    for c in range(NUM_LABELS):
        test_blend[:, c] = (1-opt_w_pc[c]) * mbert_test[:, c] + opt_w_pc[c] * svc_test_probs[:, c]
else:
    test_blend = opt_w[0] * mbert_test + opt_w[1] * svc_test_probs

print(f'\n  MB OOF F1:      {oof_f1:.4f}')
print(f'  SVC OOF F1:     {svc_oof_f1:.4f}')
print(f'  Blended OOF F1: {oof_blend_f1:.4f}')

print(f'\nTest prediction distribution:')
print(f'  {"Class":10s}  {"Fold":>6}  {"Upload":>6}  {"SVC":>6}  {"Blend":>6}')
fp = test_probs.argmax(1); up = full_test_probs.argmax(1)
sp = svc_test_probs.argmax(1); bp = test_blend.argmax(1)
for i, name in LABEL_MAP.items():
    print(f'  {name:10s}  {(fp==i).sum():6d}  {(up==i).sum():6d}  {(sp==i).sum():6d}  {(bp==i).sum():6d}')

In [ ]:
# ============================================================
# THRESHOLD + SUBMISSION
# ============================================================
def threshold_sweep(probs, labels, n_classes=6, steps=60, passes=8):
    best_mult = np.ones(n_classes)
    best_f1 = f1_score(labels, probs.argmax(1), average='macro')
    for _ in range(passes):
        improved = False
        for cls in range(n_classes):
            for m in np.linspace(0.60, 1.50, steps):
                trial = best_mult.copy(); trial[cls] = m
                f1 = f1_score(labels, (probs * trial).argmax(1), average='macro')
                if f1 > best_f1 + 1e-6:
                    best_f1, best_mult, improved = f1, trial.copy(), True
        if not improved: break
    return best_mult, best_f1

thresholds, thresh_f1 = threshold_sweep(oof_blend, y_all)
print(f'OOF F1 after threshold: {thresh_f1:.4f}  (was {oof_blend_f1:.4f})')
print(f'Multipliers: {dict(zip(LABEL_MAP.values(), thresholds.round(3)))}')

final_preds = (test_blend * thresholds).argmax(1)

id_col = (test_df['ID'] if 'ID' in test_df.columns
          else test_df['id'] if 'id' in test_df.columns
          else np.arange(len(test_df)))
submission = pd.DataFrame({'ID': id_col, 'LABEL': final_preds})
submission.to_csv('submission.csv', index=False)

print(f'\nSubmission saved!')
print(f'\nPrediction distribution:')
for i, name in LABEL_MAP.items():
    count = (final_preds == i).sum()
    print(f'  {name}: {count} ({100*count/len(final_preds):.1f}%)')

blend_type = 'per-class' if use_perclass else 'global'
print(f'\n{"="*50}')
print(f'SUMMARY')
print(f'  MB OOF F1:          {oof_f1:.4f}')
print(f'  SVC OOF F1:         {svc_oof_f1:.4f}')
print(f'  Blended OOF F1:     {oof_blend_f1:.4f}  ({blend_type})')
print(f'  + threshold:        {thresh_f1:.4f}')
print(f'  Fold/Upload alpha:  {ALPHA}')
print(f'  DRW cap:            {DRW_CAP}x')
print(f'  SWA:                top-{TOP_K_CKPTS}')
print(f'  Total time:         {(time.time()-t_start)/60:.1f} min')
print(f'{"="*50}')